In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
from plotnine import *

from tqdm import tqdm
from sklearn.metrics import r2_score


## Save results

In [ ]:
config_path = '../../run_config_local.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

traits=config.get('traits')
# traits

In [ ]:
funcrvp_path = f"/s/project/geno2pheno/funcrvp/paper_revisions/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_val"

model_dir_list = []
for s in [0.05, 0.25, 0.375]:
    model_dir_list.append(funcrvp_path + str(s) + '_filteredv3')

model_dir_list

In [ ]:
skip_list = []
for model_dir in tqdm(model_dir_list):
    beta_trait_list = []
    pred_trait_list = []
    for trait in tqdm(traits):
        try:
            # Load the predictions
            beta_file = os.path.join(model_dir, f"{trait}_betas.pq")
            beta_trait_list.append(pd.read_parquet(beta_file))

            pred_file = os.path.join(model_dir, f"{trait}_phenopred.pq")  
            pred_trait_list.append(pd.read_parquet(pred_file))
        
        except FileNotFoundError:
            skip_list.append(f"{trait}: {model_dir}")

    # Concatenate the dataframes
    beta_df = pd.concat(beta_trait_list, axis=0)
    pred_df = pd.concat(pred_trait_list, axis=0).reset_index()
    # Save the concatenated dataframes to parquet files
    beta_df.to_parquet(os.path.join(model_dir, "all_traits_betas.pq"))
    pred_df.to_parquet(os.path.join(model_dir, "all_traits_phenopred.pq"))

print(skip_list)

## Read results

In [ ]:
funcrvp_path = f"/s/project/geno2pheno/funcrvp/paper_revisions/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_val"

beta_df_list = []
pred_df_list = []

for s in tqdm([0.05, 0.25, 0.375]):
    # Save the concatenated dataframes to parquet files
    temp_b = pd.read_parquet(os.path.join(funcrvp_path + str(s) + '_filteredv3', "all_traits_betas.pq"))
    temp_b['val_size'] = s
    beta_df_list.append(temp_b)
    temp_p = pd.read_parquet(os.path.join(funcrvp_path + str(s) + '_filteredv3', "all_traits_phenopred.pq"))
    temp_p['val_size'] = s
    pred_df_list.append(temp_p)

# Add the actual model (val_size = 0.1)
base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions'
temp_b = pd.read_parquet(f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3/all_traits_betas.pq")
temp_b['val_size'] = 0.1
beta_df_list.append(temp_b)

temp_p = pd.read_parquet(f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3/all_traits_phenopred.pq")
temp_p['val_size'] = 0.1
pred_df_list.append(temp_p)
# pd.concat(beta_df_list).to_parquet(f'{pdir}/funcrvp_betas_all_sampling.pq')
# pd.concat(pred_df_list).to_parquet(f'{pdir}/funcrvp_phenopred_all_sampling.pq')

fg = pd.concat(beta_df_list)
fp = pd.concat(pred_df_list)
fp

In [ ]:
fp[['val_size', 'trait']].value_counts()

In [ ]:
fp_r2 = pd.DataFrame(fp.groupby(['val_size', 'trait']).apply(lambda group: r2_score(group['trait_measurement'], group['best_prediction']))).reset_index()
fp_r2.columns = ['val_size', 'trait', 'funcrvp_r2']
fp_r2

In [ ]:
lm_cov = pd.read_parquet("/s/project/geno2pheno/funcrvp/paper_revisions/predictions/all_traits_covariates_only_phenopred_filteredv3.pq")
lm_cov

In [ ]:
cov_r2 = pd.DataFrame(lm_cov.groupby(['trait']).apply(lambda group: r2_score(group['trait_measurement'], group['best_prediction']))).reset_index()
cov_r2.columns = ['trait', 'cov_r2']
cov_r2

In [ ]:
plot_df = fp_r2.merge(cov_r2, on='trait')
plot_df['funcrvp_rel_del_r2'] = (plot_df['funcrvp_r2'] - plot_df['cov_r2'])/plot_df['cov_r2']
plot_df

In [ ]:
from statsmodels.formula.api import ols

# Fit a linear model
model = ols('funcrvp_rel_del_r2 ~ val_size', data=plot_df).fit()

(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='val_size')) +
    geom_point() +
    stat_smooth(method='lm') +
    xlab('Validation set size') +
    ylab('FuncRVP relative delta R^2') +
    annotate("text", x=0.2, y=max(plot_df['funcrvp_rel_del_r2']), 
        label=f"Slope: {model.params['val_size']:.4f}\nP-value: {model.pvalues['val_size']:.4e}", 
        ha='center', va='top', size=10, color='red') +
    theme_bw()
)

In [ ]:
plot_df['val_size'] = pd.Categorical(plot_df['val_size'], categories=plot_df['val_size'].unique())
(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='val_size')) +
    geom_boxplot() +
    xlab('Validation set size') +
    ylab('FuncRVP relative delta R^2') +
    theme_bw()
)

In [ ]:
import itertools
from scipy.stats import mannwhitneyu
# Get all unique validation sizes
val_sizes = plot_df['val_size'].unique()

print("Performing pairwise Wilcoxon (Mann-Whitney U) tests across all val_size combinations:")
print("(Note: P-values are unadjusted for multiple comparisons)")

# Iterate through all unique pairs of val_size categories
for size1, size2 in itertools.combinations(val_sizes, 2):
    data1 = plot_df[plot_df['val_size'] == size1]['funcrvp_rel_del_r2'].dropna()
    data2 = plot_df[plot_df['val_size'] == size2]['funcrvp_rel_del_r2'].dropna()

    # Ensure there's enough data for the test
    if len(data1) > 0 and len(data2) > 0:
        statistic, p_value = mannwhitneyu(data1, data2, alternative='two-sided')
        print(f"  {size1} vs {size2}: Statistic = {statistic:.4f}, p-value = {p_value:.4f}")
    else:
        print(f"  Skipping {size1} vs {size2}: Not enough data in one or both groups.")
